<h1>Chapter 2 - Generation Models</h1>
<i>Choosing the generation model for your RAG system.</i>

<a href="https://learning.oreilly.com/library/view/rag-with-python/9798341600553/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/polzerdo55862/RAG-with-Python-Cookbook"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/polzerdo55862/RAG-with-Python-Cookbook/blob/main/ch01_RAG_intro/rag_basics.ipynb)

---

This notebook is for Chapter 3 of the [RAG with Python Cookbook](https://learning.oreilly.com/library/view/rag-with-python/9798341600553/) book by [Dominik Polzer](https://www.linkedin.com/in/polzerdo/).

---

<a href="https://learning.oreilly.com/library/view/rag-with-python/9798341600553/">
  <img src="https://raw.githubusercontent.com/polzerdo55862/RAG-with-Python-Cookbook/main/rag_cookbook.png" width="350" />
</a>


In [4]:
!pip install openai

### Load secrets

If you run this code in Google Colab, save your OpenAI API key in the secrets and access it by

In [6]:
from google.colab import userdata
import os

api_key = userdata.get("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY not found in Colab Secrets")

os.environ["OPENAI_API_KEY"] = api_key

## 1. OpenAI Chat Completions

In [7]:
from openai import OpenAI

def ask_with_context(context, question):
    client = OpenAI()

    messages = [
        {"role": "system", "content": "Answer based only on the provided context."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion:\n{question}"},
    ]

    response = client.chat.completions.create(
        model="gpt-5", messages=messages  # Using latest model
    )

    return response.choices[0].message.content


# Usage
context = "RAG stands for Retrieval-Augmented Generation."
question = "What does RAG stand for?"
answer = ask_with_context(context, question)
print(answer)

Retrieval-Augmented Generation.


## 2. OpenAI Whisper Speech-to-Text

In [8]:
import httpx
from openai import OpenAI

client = OpenAI(http_client=httpx.Client(verify=False))

with open(
    "..\\datasets\\audio_files\\LJ037-0171.wav",
    "rb",
) as audio_file:
    transcript = client.audio.transcriptions.create(
        model="gpt-4o-mini-transcribe",  # latest speech model name
        file=audio_file,
    )

print(transcript.text)

FileNotFoundError: [Errno 2] No such file or directory: '..\\datasets\\audio_files\\LJ037-0171.wav'

## 3. Anthropic Claude Example

In [9]:
from anthropic import Anthropic
import os

client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

response = client.messages.create(
    model="claude-sonnet-4-5",
    max_tokens=200,
    messages=[
        {
            "role": "user",
            "content": "Explain how vector databases work in simple terms.",
        }
    ],
)

print(response.content[0].text)

ModuleNotFoundError: No module named 'anthropic'

## 4. Gemini API Example using the OpenAI SDK

In [ ]:
"""
pip install openai
"""

import os
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

resp = client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=[{"role": "user", "content": "What is the capital of France?"}],
)

print(resp.choices[0].message.content)

## 5. Deploy local LLMs using Ollama

In [ ]:
"""
Running Local LLMs with Ollama
Shows how to use locally hosted models via Ollama with the OpenAI SDK

pip install openai
"""

from openai import OpenAI

# Point the client to your local Ollama server
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",  # Ollama doesn't require a real key, but the SDK expects one
)

# Send a chat completion request
response = client.chat.completions.create(
    model="quen3:4b",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is retrieval-augmented generation?"},
    ],
)

print(response.choices[0].message.content)

## 6. Pydantic Structured Output

In [ ]:
from pydantic import BaseModel
from openai import OpenAI


class Person(BaseModel):
    first_name: str
    last_name: str
    email: str
    age: int


client = OpenAI()

text = "My name is Sarah Johnson, I'm 34 years old. Email me at sarah.j@example.com"

response = client.beta.chat.completions.parse(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": "Extract person information."},
        {"role": "user", "content": text},
    ],
    response_format=Person,
)

person = response.choices[0].message.parsed
print(person.first_name)
print(person.email)
print(person.age)